# 01 - Ingest Raw Data: Retail Analytics

**Project:** Retail Analytics & Product Dimension History

## What this notebook does
Documents the source and acquisition method for the raw retail star-schema
data, and confirms the files landed correctly in the raw_data Volume before
bronze ingestion runs.

## Source
Kaggle: retail-sales-dataset (buharishehu)
https://www.kaggle.com/datasets/buharishehu/retail-sales-dataset

Source data was delivered as a single Excel workbook with four tabs
(Customers, Products, Stores, Transactions). Each tab was exported as a
separate CSV file to keep this project's ingestion pattern consistent with
Healthcare, P&C, and Auto Insurance (Autoloader-based, one bronze table per
source file), rather than introducing a separate Excel-parsing mechanism for
what is fundamentally a one-time source extract.

## Output
Raw CSVs confirmed present in `/Volumes/main/retail_analytics/raw_data/`

In [0]:
files = dbutils.fs.ls("/Volumes/main/retail_analytics/raw_data")
for f in files:
    print(f"{f.name:20} {f.size / 1024:.1f} KB")

Customers.csv        11.2 KB
Products.csv         2.7 KB
Stores.csv           0.2 KB
Transactions.csv     248.8 KB


In [0]:
for filename in ["Customers.csv", "Products.csv", "Stores.csv", "Transactions.csv"]:
    path = f"/Volumes/main/retail_analytics/raw_data/{filename}"
    df = spark.read.option("header", "true").option("inferSchema", "true").csv(path)
    print(f"=== {filename} ===")
    df.printSchema()
    print(f"Row count: {df.count()}")
    print()

=== Customers.csv ===
root
 |-- CustomerID: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BirthDate: date (nullable = true)
 |-- City: string (nullable = true)
 |-- JoinDate: date (nullable = true)

Row count: 200

=== Products.csv ===
root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CostPrice: double (nullable = true)

Row count: 50

=== Stores.csv ===
root
 |-- StoreID: string (nullable = true)
 |-- StoreName: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Region: string (nullable = true)

Row count: 5

=== Transactions.csv ===
root
 |-- TransactionID: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- S

## Data quality note
All four files confirmed with clean, correctly-typed columns - no placeholder
values, no malformed rows, no encoding issues found (unlike prior projects'
raw files). Star schema confirmed: Products, Customers, and Stores are
dimension tables; Transactions is the fact table referencing all three via
ProductID, CustomerID, and StoreID.

## Row counts
- Customers: 200
- Products: 50
- Stores: 5
- Transactions: 5,000